In [ ]:
from pyspark.sql import SparkSession
from commons import get_raw_data, filter_southern_region

spark = SparkSession.builder.appName("IcebergLab") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "../data/iceberg") \
    .getOrCreate()

: 

In [ ]:
df_vendas = get_raw_data(spark, "vendas.csv")
df_clientes = get_raw_data(spark, "clientes.csv")
df_sul = filter_southern_region(df_vendas.join(df_clientes, "id_cliente"))

In [ ]:
df_sul.write.format("iceberg").mode("overwrite").saveAsTable("local.vendas_sul")

In [ ]:
spark.sql("UPDATE local.vendas_sul SET status = 'devolvido' WHERE id_venda = 5")
spark.sql("DELETE FROM local.vendas_sul WHERE id_venda = 2")
spark.sql("SELECT * FROM local.vendas_sul").show()